# 02 — Data Quality Analysis

This notebook applies reusable validation checks to the normalized event dataset and exports an auditable cleaned dataset.

## Coverage
- Required schema and type validation
- Missing values and duplicate records
- Timestamp and stage checks
- Duplicate customer events
- Outlier and category checks
- Cleaning summary and cleaned-data export

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "ecommerce_funnel_events.csv"
CLEAN_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
REPORTS_PATH = ROOT / "reports"
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

raw = pd.read_csv(DATA_PATH)
raw["event_timestamp"] = pd.to_datetime(raw["event_timestamp"], utc=True, errors="coerce")

required = {"event_id", "user_id", "session_id", "event_timestamp", "campaign", "channel", "device", "stage", "cart_value", "revenue"}
missing_columns = sorted(required - set(raw.columns))

checks = {}
checks["missing_required_columns"] = len(missing_columns)
checks["missing_cells"] = int(raw.isna().sum().sum())
checks["duplicate_exact_rows"] = int(raw.duplicated().sum())
checks["duplicate_event_ids"] = int(raw["event_id"].duplicated().sum()) if "event_id" in raw else 0
checks["invalid_timestamps"] = int(raw["event_timestamp"].isna().sum())
checks["invalid_stages"] = int((~raw["stage"].isin(["product_view", "cart", "purchase"])).sum()) if "stage" in raw else len(raw)

missing = raw.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(raw) * 100).round(3)
display(missing[missing["missing_count"] > 0])

if "stage" in raw:
    fig, ax = plt.subplots(figsize=(8, 4))
    raw["stage"].value_counts(dropna=False).plot(kind="bar", ax=ax, color="#4C78A8")
    ax.set_title("Observed Funnel Stage Categories")
    ax.set_ylabel("Rows")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

clean = raw.drop_duplicates().copy()
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

clean = clean.loc[clean["event_timestamp"].notna()].copy()
clean = clean.loc[clean["stage"].isin(["product_view", "cart", "purchase"])].copy()
clean["event_id"] = clean["event_id"].astype(str).str.strip()
clean["user_id"] = clean["user_id"].astype(str).str.strip()
clean = clean.drop_duplicates(subset=["event_id"])
clean = clean.sort_values(["user_id", "event_timestamp", "stage"])
clean.to_csv(CLEAN_PATH, index=False)

quality_summary = pd.DataFrame([
    {"check": name, "affected_count": value, "status": "PASS" if value == 0 else "REVIEW"}
    for name, value in checks.items()
])
quality_summary["raw_rows"] = len(raw)
quality_summary["clean_rows"] = len(clean)
quality_summary.to_csv(REPORTS_PATH / "data_quality_summary.csv", index=False)

print("Missing required columns:", missing_columns)
print("Raw rows:", len(raw))
print("Clean rows:", len(clean))
print("Rows excluded:", len(raw) - len(clean))
display(quality_summary)
print("Clean dataset:", CLEAN_PATH)